# Lecture 9 · Notebook 2 — Retrieval-augmented answering with citations
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture09/lecture09_2_rag_with_citations.ipynb)

**Companion to [Chapter 9 of the video lecture](https://nlp-course-cuhksz.github.io/lecture09.html#p9-s132)** (community evidence-based medicine and an "evidence API" every agent
can retrieve from) and to the four properties of **trustworthy medical AI** in [Chapter 8](https://nlp-course-cuhksz.github.io/lecture09.html#p8-s108): evidence-grounded,
calibrated, auditable, bounded. A model that answers from memory cannot show where an answer comes from. A model that **retrieves**
evidence first and **cites** it can be checked. In this notebook you will:

1. build a small evidence base of real PubMed abstracts (from **PubMedQA**, Jin et al., 2019),
2. write a **BM25** retriever in a few lines and measure how often it finds the right abstract,
3. compare a 0.5B model answering research questions **closed-book**, with **retrieved** evidence, and with the **true** abstract, and
4. make it answer **with numbered citations**, then *check* those citations automatically, and let it **abstain** when nothing relevant is found.

Runs on a free CPU runtime in about 10 minutes.

> ⚠️ **Not medical advice.** This notebook studies how language models *behave* on medical test questions. The model is tiny and often
> wrong; nothing it writes here is a diagnosis or a treatment recommendation. For health questions, ask a qualified clinician.

In [ ]:
!pip -q install -U transformers

In [ ]:
import torch, json, random, re, time, collections
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # ~1 GB; try "Qwen/Qwen2.5-1.5B-Instruct" on a T4 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
print("device:", device, "| parameters:", f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M")

## 0 · The evidence base: PubMedQA abstracts
PubMedQA's 1,000 expert-labelled items each pair a research question (often an article's title) with a PubMed abstract and a
**yes / no / maybe** answer. We pool every abstract's paragraphs, including its **conclusion**, into one corpus of a few thousand
passages, a miniature "evidence base". For each question the retriever must find its own abstract among all the others.

In [ ]:
import requests
URL = "https://raw.githubusercontent.com/pubmedqa/pubmedqa/master/data/ori_pqal.json"
data = requests.get(URL, timeout=60).json()
qa = [{"pmid": k, "q": v["QUESTION"], "label": v["final_decision"], "year": v.get("YEAR")} for k, v in data.items()]
passages = []
for k, v in data.items():
    passages += [{"id": f"PMID {k} §{i+1}", "pmid": k, "text": c} for i, c in enumerate(v["CONTEXTS"])]
    passages.append({"id": f"PMID {k} §concl", "pmid": k, "text": v["LONG_ANSWER"]})
print(len(qa), "questions,", len(passages), "passages")
print("labels:", dict(collections.Counter(x["label"] for x in qa)))
print("\nexample:", qa[0]["q"], "->", qa[0]["label"])
print(passages[0]["id"], passages[0]["text"][:300], "...")

## 1 · A BM25 retriever in a dozen lines
BM25 is the classic lexical ranking function behind most search engines: a word counts more if it is rare in the corpus (IDF) and
frequent in the passage, with diminishing returns and a length normalisation. No training, no GPU, and every score can be explained,
which is an advantage when the results must be audited.

In [ ]:
STOP = set("a an the of in on for to and or with without is are was were be been by from at as that this these those than does do did not "
           "can could which what who whom whose how why when there their its it into after before between during among we our".split())

def terms(s):
    return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP and len(w) > 1]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.toks = [terms(d) for d in docs]
        self.k1, self.b = k1, b
        self.len = np.array([len(t) for t in self.toks]); self.avg = self.len.mean()
        self.tf = [collections.Counter(t) for t in self.toks]
        df = collections.Counter(w for t in self.toks for w in set(t))
        n = len(docs)
        self.idf = {w: np.log(1 + (n - d + 0.5) / (d + 0.5)) for w, d in df.items()}
    def scores(self, query):
        s = np.zeros(len(self.toks))
        for w in set(terms(query)):
            if w not in self.idf: continue
            f = np.array([tf.get(w, 0) for tf in self.tf])
            s += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len / self.avg))
        return s
    def search(self, query, k=3):
        s = self.scores(query); top = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in top]

t0 = time.time()
bm25 = BM25([p["text"] for p in passages])
print(f"index built in {time.time()-t0:.1f}s")
for i, s in bm25.search(qa[0]["q"]):
    print(f"{s:6.2f}  {passages[i]['id']:18s} {passages[i]['text'][:90]}...")

**How good is the retriever?** For 200 questions, check whether the top-k passages include one from the question's own abstract
(*recall@k*). This is the first thing to measure in any RAG system: the generator cannot cite evidence the retriever never found.

In [ ]:
random.seed(0)
sample = random.sample(qa, 200)
for k in (1, 3, 5, 10):
    hit = np.mean([any(passages[i]["pmid"] == x["pmid"] for i, _ in bm25.search(x["q"], k)) for x in sample])
    print(f"recall@{k:<2d} {hit:.0%}")
top1 = np.array([bm25.search(x["q"], 1)[0][1] for x in sample])
print(f"best-passage BM25 score for real questions: 1st percentile {np.percentile(top1, 1):.1f}, median {np.median(top1):.1f}")

## 2 · Closed-book vs. retrieved evidence vs. the true abstract
We now ask the model to answer *yes* or *no* and read its choice from the next-token probabilities, as in Notebook 1. (We drop the 11%
of questions labelled *maybe*: when we offered "maybe" as an option, this small model hedged with "maybe" on every single question. A
model that always hedges looks safe and is useless.) Three settings:

* **closed-book**: only the question,
* **RAG**: the question plus the top-3 BM25 passages,
* **oracle**: the question plus its own abstract's conclusion (what perfect retrieval of the key passage would give).

In [ ]:
ANS = ["yes", "no"]
# the model may start its answer with "yes" or "Yes": add up both spellings
ANS_IDS = [[tok.encode(v, add_special_tokens=False)[0] for v in (a, a.capitalize())] for a in ANS]
SYSTEM = "You answer biomedical research questions using the evidence provided. You do not give medical advice."

def evidence_block(pids):
    return "\n".join(f"[{j+1}] {passages[i]['text']}" for j, i in enumerate(pids))

def yn_prompt(question, pids):
    ctx = ("Evidence:\n" + evidence_block(pids) + "\n\n") if pids else ""
    user = ctx + "Question: " + question + "\nAnswer with one word: yes or no."
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def yn(question, pids):
    x = tok(yn_prompt(question, pids), return_tensors="pt", truncation=True, max_length=3000).to(device)
    logits = model(**x).logits[0, -1].float()
    return ANS[int(torch.stack([logits[ids].logsumexp(0) for ids in ANS_IDS]).argmax())]

concl = {p["pmid"]: i for i, p in enumerate(passages) if p["id"].endswith("concl")}

evalset = [x for x in sample if x["label"] != "maybe"][:60]
E = len(evalset)
res = {"closed-book": [], "RAG (BM25 top-3)": [], "oracle conclusion": []}
preds = {k: [] for k in res}
t0 = time.time()
for x in evalset:
    for k, pids in (("closed-book", []), ("RAG (BM25 top-3)", [i for i, _ in bm25.search(x["q"], 3)]), ("oracle conclusion", [concl[x["pmid"]]])):
        a = yn(x["q"], pids)
        preds[k].append(a); res[k].append(a == x["label"])
print(f"[{time.time()-t0:.0f}s]")
majority = max(collections.Counter(x["label"] for x in evalset).values()) / E
print("gold labels:", dict(collections.Counter(x["label"] for x in evalset)))
for k, v in res.items(): print(f"{k:18s} accuracy {np.mean(v):.0%}   predictions {dict(collections.Counter(preds[k]))}")
print(f"{'always majority':18s} {majority:.0%}")
plt.figure(figsize=(6, 3))
plt.bar(list(res) + ["majority"], [np.mean(v) for v in res.values()] + [majority], color=["#999999", "#4c72b0", "#55a868", "#dddddd"])
plt.ylim(0, 1); plt.ylabel("accuracy"); plt.title(f"PubMedQA, {E} questions, {MODEL_ID.split('/')[-1]}"); plt.show()

Closed-book, the 0.5B model knows almost nothing about these specific studies and mostly says "yes", so it lands near the majority
baseline. Given the right evidence it does much better: the reader does not need to *know* the answer, only to *read* it. Retrieval sits in
between, because the top-3 passages do not always include the conclusion. With 60 questions the error bars are about ±12 points, so
treat small differences as noise.

Just as important: with retrieval, every answer now comes *with the passages it was based on*, which a reader can check. That is what the
next section uses.

## 3 · Answers with numbered citations, and a citation checker
We ask for a short answer that cites the evidence as `[1]`, `[2]`, … and then check the citations automatically:

* **valid**: every cited number refers to a passage that was actually shown,
* **on-source**: the cited passage comes from the abstract the question was taken from (we know it here; in real life you would
  check whether the passage *entails* the claim),
* **uncited**: the answer cites nothing, so it cannot be audited.

We also add a simple **abstention rule** (slide 108, *calibrated* and *bounded*): if the best BM25 score is low, the system says it found no
evidence instead of answering from memory.

In [ ]:
MIN_SCORE = 12.0    # below this best-passage score we abstain (99% of real PubMedQA questions score higher); tune it in Exercise 2

@torch.no_grad()
def answer_with_citations(question, k=3, max_new_tokens=90):
    hits = bm25.search(question, k)
    if hits[0][1] < MIN_SCORE:
        return {"text": "I could not find evidence for this question in the evidence base, so I will not answer it.", "pids": [], "abstained": True}
    pids = [i for i, _ in hits]
    user = ("Evidence:\n" + evidence_block(pids) + "\n\nQuestion: " + question +
            "\nStart with yes or no. Then explain in one or two sentences, citing the evidence by number like [1].")
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    x = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt").to(device)
    out = model.generate(**x, max_new_tokens=max_new_tokens, do_sample=False)
    return {"text": tok.decode(out[0, x["input_ids"].shape[1]:], skip_special_tokens=True).strip(), "pids": pids, "abstained": False}

def check_citations(ans, pmid):
    cited = sorted({int(c) for c in re.findall(r"\[(\d+)\]", ans["text"])})
    valid = [c for c in cited if 1 <= c <= len(ans["pids"])]
    on_source = [c for c in valid if passages[ans["pids"][c - 1]]["pmid"] == pmid]
    return {"cited": cited, "invalid": [c for c in cited if c not in valid], "on_source": on_source, "uncited": not cited}

for x in evalset[:3]:
    a = answer_with_citations(x["q"])
    chk = check_citations(a, x["pmid"])
    print("Q:", x["q"], f"   (gold: {x['label']})")
    print("A:", a["text"])
    print("   check:", chk)
    for c in chk["cited"]:
        if 1 <= c <= len(a["pids"]):
            p = passages[a["pids"][c - 1]]
            print(f"   [{c}] {p['id']}: {p['text'][:140]}...")
    print()

Now over 30 questions: how often does the model cite at all, cite something that exists, and cite the right abstract?

In [ ]:
stats = collections.Counter()
t0 = time.time()
for x in evalset[:30]:
    a = answer_with_citations(x["q"])
    if a["abstained"]: stats["abstained"] += 1; continue
    c = check_citations(a, x["pmid"])
    stats["answered"] += 1
    stats["uncited"] += c["uncited"]
    stats["has invalid citation"] += bool(c["invalid"])
    stats["cites own abstract"] += bool(c["on_source"])
print(f"[{time.time()-t0:.0f}s]", dict(stats))

Now two questions the evidence base cannot answer. The retriever always returns *something*; the abstention rule decides whether to use it.

In [ ]:
for q in ["Which football club won the 1998 World Cup final?", "What is the capital of Australia?"]:
    hits = bm25.search(q, 1)
    print(f"best BM25 score {hits[0][1]:.1f} | {q}")
    print(f"  top passage: {passages[hits[0][0]]['id']}: {passages[hits[0][0]]['text'][:150]}...")
    print("  ->", answer_with_citations(q)["text"][:300], "\n")

The first question is caught: nothing in the corpus matches well, so the system declines. The second slips through: a rare word such as
*Australia* appears in some abstract, the BM25 score is high, and the model is handed irrelevant evidence. A lexical score measures word
overlap, not whether the passage answers the question. And even a valid, on-source citation does not guarantee that the sentence citing
it is true. That gap is exactly
why the lecture's evidence platform keeps **experts in the loop** who edit and sign the content (slides 132–134), and why the dental
report on slide 155 shows its references to the user.

## Exercises
1. Replace BM25 with a dense retriever (e.g. `sentence-transformers/all-MiniLM-L6-v2` or `BAAI/bge-small-en-v1.5`, mean-pooled embeddings
   and cosine similarity). Does recall@3 go up? Try *hybrid* retrieval: add the two normalised scores.
2. Plot the best BM25 score for the 200 sample questions and for 20 off-topic questions you write yourself. Where would you put
   `MIN_SCORE`? What does a too-high threshold cost?
3. Write a stricter citation checker: for each cited sentence, ask the model *"Does passage [n] support this sentence? yes/no"*. How often
   does a valid, on-source citation still not support the claim?
4. Restrict the evidence base to abstracts published before a certain `year` and ask about later ones. This is the *stale evidence*
   problem behind the "AI-native UpToDate" idea.

**Next:** [Notebook 3 — an LLM judge, position bias, and the order-swapping trick](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture09/lecture09_3_llm_judge_bias.ipynb)